In [2]:
import sys
import psycopg2
import pandas as pd

sys.path.append("..")

from Keys.GCalendarAPIKey import GCalendarAPI
from Keys.PostGresKey import POSTGRES_URL

from googleapiclient.discovery import build

# =========================================
# CONFIG
# =========================================

TARGET_YEAR = 2026

CALENDAR_ID = (
    "en.eg.official#holiday@group.v.calendar.google.com"
)

# =========================================
# GOOGLE CALENDAR
# =========================================

service = build(
    "calendar",
    "v3",
    developerKey=GCalendarAPI
)

events = service.events().list(

    calendarId=CALENDAR_ID,

    timeMin=f"{TARGET_YEAR}-01-01T00:00:00Z",

    timeMax=f"{TARGET_YEAR}-12-31T23:59:59Z",

    singleEvents=True,

    orderBy="startTime"

).execute()

rows = []

for event in events.get("items", []):

    holiday_date = event["start"].get("date")

    if not holiday_date:
        continue

    rows.append({

        "holiday_date": holiday_date,

        "holiday_name": event["summary"]

    })

df = pd.DataFrame(rows)

df = df.drop_duplicates(
    subset=["holiday_date"]
)

print(df)

print(
    f"\nTotal Holidays: {len(df)}"
)

# =========================================
# POSTGRES
# =========================================

conn = psycopg2.connect(
    POSTGRES_URL
)

cursor = conn.cursor()

for _, row in df.iterrows():

    cursor.execute(

        """

        INSERT INTO Official_Holidays

        (

            Holiday_Date,
            Holiday_Name,
            Source

        )

        VALUES

        (

            %s,
            %s,
            %s

        )

        ON CONFLICT
        (Holiday_Date)

        DO UPDATE

        SET

            Holiday_Name =
            EXCLUDED.Holiday_Name,

            Source =
            EXCLUDED.Source

        """,

        (

            row["holiday_date"],

            row["holiday_name"],

            "Google Calendar"

        )

    )

conn.commit()

cursor.close()
conn.close()

print(
    "\n2026 Holidays Loaded Successfully"
)

   holiday_date                            holiday_name
0    2026-01-07                    Coptic Christmas Day
1    2026-01-25               Revolution Day January 25
2    2026-01-29   Day off for Revolution Day January 25
3    2026-03-19                     Eid el Fitr Holiday
4    2026-03-20                             Eid el Fitr
5    2026-03-21                     Eid el Fitr Holiday
6    2026-03-22                     Eid el Fitr Holiday
7    2026-03-23                     Eid el Fitr Holiday
8    2026-04-13                         Spring Festival
9    2026-04-25                    Sinai Liberation Day
10   2026-05-01                               Labor Day
11   2026-05-26                              Arafat Day
12   2026-05-27                             Eid al-Adha
13   2026-05-28                     Eid al-Adha Holiday
14   2026-05-29                     Eid al-Adha Holiday
15   2026-05-30                     Eid al-Adha Holiday
16   2026-05-31                     Eid al-Adha 